# NetFlow dataset exploration

Data come directly from UQ. Run the fetch scripts described in README.md first. Raw data are ignored by Git; this notebook displays aggregate counts rather than endpoint identifiers.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/network_detector.py').exists())


In [ ]:
counts = []
for source in ['NF-ToN-IoT', 'NF-BoT-IoT']:
    totals = pd.Series(dtype='int64')
    for chunk in pd.read_csv(ROOT / f'data/raw/{source}.csv', usecols=['Attack'], chunksize=100_000):
        totals = totals.add(chunk.Attack.value_counts(), fill_value=0)
    counts.append(totals.rename(source))
class_counts = pd.concat(counts, axis=1).fillna(0).astype(int)
class_counts


In [ ]:
class_counts.plot.barh(figsize=(10, 6), title='Published-source flow counts by attack family')


## Split and cleaning audit

The final holdouts retain repeated and conflicting feature patterns. Only fitting records are deduplicated. Host-pair separation still permits individual hosts to occur in different splits.


In [ ]:
report = json.loads((ROOT / 'models/training_report.json').read_text())
pd.DataFrame(report['split_counts']).T


In [ ]:
report['cleaning']
